In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from typing import List
import re

In [18]:
df = pd.read_csv('/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/processed/cases.csv')
df

,case_id,nomor,tanggal_register,tanggal_dibacakan,ringkasan_fakta,pasal,amar,catatan_amar,text_full
0,case_001,Putusan PN PALU Nomor 116/Pid.B/2025/PN Pal Ta...,28 April 2025,19 Juni 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 338 KUHP Jo Pasal ; Pasal 65 ayat ; Pasa...,Lain-lain,MENGADILI:1. Menyatakan Terdakwa Moh. Zakir A...,Direktori Putusan Mahkamah Agung Republik Indo...
1,case_002,Putusan PN KAYUAGUNG Nomor 97/Pid.B/2025/PN Ka...,10 Maret 2025,22 Mei 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 222 Undang; Pasal 8 ayat ; Pasal 187 Und...,Lain-lain,MENGADILI: Menyatakan Terdakwa FAUZAN ROHMAT B...,Direktori Putusan Mahkamah Agung Republik Indo...
2,case_003,Putusan PN Cikarang Nomor 10/Pid.B/2025/PN Ckr...,16 Januari 2025,4 Juni 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 10 KUHP bahwa pembebanan biaya perkara k...,Lain-lain,MENGADILI: 1. Menyatakan Terdakwa HAGISTIKO PR...,Direktori Putusan Mahkamah Agung Republik Indo...
3,case_004,Putusan PN NABIRE Nomor 23/Pid.B/2025/PN Nab T...,13 Maret 2025,19 Juni 2025,Direktori Putusan Mahkamah Agung Republik Indo...,NaN,Lain-lain,MENETAPKAN:1.Menyatakan Penununtutan Jaksa Pen...,Direktori Putusan Mahkamah Agung Republik Indo...
4,case_005,Putusan PN PALANGKARAYA Nomor 50/Pid.B/2025/PN...,27 Februari 2025,19 Mei 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 84 ayat ; Pasal 28 ayat ; Pasal 6 ayat ;...,Lain-lain,MENGADILI: Menyatakan Terdakwa Muhammad Haryon...,Direktori Putusan Mahkamah Agung Republik Indo...
...,...,...,...,...,...,...,...,...,...
304,case_305,Putusan PN SAMPANG Nomor 202/Pid.B/2024/PN Spg...,18 Nopember 2024,8 Januari 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 338 KUHP termasukdelik materil karena me...,Lain-lain,MENGADILI: Menyatakan Terdakwa FAUZUN Bin H. M...,Direktori Putusan Mahkamah Agung Republik Indo...
305,case_306,Putusan PN KAYUAGUNG Nomor 515/Pid.B/2024/PN K...,25 Oktober 2024,23 Januari 2025,NaN,NaN,Lain-lain,MENGADILI: Menyatakan TerdakwaDARMIZI BIN SULA...,NaN
306,case_307,Putusan PN PELALAWAN Nomor 218/Pid.B/2024/PN P...,16 Agustus 2024,2 Januari 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 359 KUHP adalah barang siapa; Pasal 359 ...,Lain-lain,MENGADILI: Menyatakan Terdakwa ABDI JAYA NEGAR...,Direktori Putusan Mahkamah Agung Republik Indo...
307,case_308,Putusan PN DONGGALA Nomor 204/Pid.B/2024/PN Dg...,26 September 2024,7 Januari 2025,Direktori Putusan Mahkamah Agung Republik Indo...,Pasal 22 ayat ; Pasal 8 ayat ; Pasal 338 jo; P...,Lain-lain,MENGADILI: Menyatakan Terdakwa Jufrianton alia...,Direktori Putusan Mahkamah Agung Republik Indo...


In [19]:
import pandas as pd
import re

# 1. Load data
cases_path = '/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/processed/cases.csv'
df = pd.read_csv(cases_path)
df['catatan_amar'] = df['catatan_amar'].fillna('')

# 2. Peta kata ke angka
angka_map = {
    'satu': '1', 'dua': '2', 'tiga': '3', 'empat': '4', 'lima': '5',
    'enam': '6', 'tujuh': '7', 'delapan': '8', 'sembilan': '9', 'sepuluh': '10',
    'sebelas': '11', 'dua belas': '12', 'tiga belas': '13', 'empat belas': '14',
    'lima belas': '15', 'enam belas': '16', 'tujuh belas': '17', 'delapan belas': '18',
    'sembilan belas': '19', 'dua puluh': '20'
}

def kata_ke_angka(text):
    for huruf, angka in angka_map.items():
        text = re.sub(rf'\b{huruf}\b', angka, text)
    return text

# 3. Normalisasi teks
def normalize_text(text):
    text = text.lower()
    text = kata_ke_angka(text)
    text = re.sub(r'\d+\s*\([^)]+\)', lambda m: re.findall(r'\d+', m.group(0))[0], text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['catatan_amar_clean'] = df['catatan_amar'].apply(normalize_text)

# 4. Ekstraksi lama hukuman (bulan)
def extract_sentence_duration(text):
    # Hukuman mati
    if 'pidana mati' in text or 'dijatuhkan pidana mati' in text:
        return 1000
    # Seumur hidup
    if 'seumur hidup' in text:
        return 999
    # Tahun dan bulan
    match = re.search(r'(\d+)\s*(tahun|th)\s*(dan)?\s*(\d+)?\s*bulan?', text)
    if match:
        tahun = int(match.group(1))
        bulan = int(match.group(4)) if match.group(4) else 0
        return tahun * 12 + bulan
    # Hanya tahun
    match = re.search(r'(\d+)\s*(tahun|th)', text)
    if match:
        return int(match.group(1)) * 12
    # Hanya bulan
    match = re.search(r'(\d+)\s*bulan', text)
    if match:
        return int(match.group(1))
    # Bebas
    if any(kw in text for kw in ['bebas', 'melepaskan', 'dilepaskan']):
        return 0
    return None

df['lama_hukuman_bulan'] = df['catatan_amar_clean'].apply(extract_sentence_duration)

# 5. Pelabelan hasil putusan
def label_putusan(row):
    text = row['catatan_amar_clean']
    hukuman = row['lama_hukuman_bulan']

    if hukuman == 1000:
        return 'hukuman_mati'
    if hukuman == 999:
        return 'seumur_hidup'
    if pd.notnull(hukuman) and hukuman > 0:
        return 'ditahan'
    if hukuman == 0:
        return 'dibebaskan'
    if 'tidak dapat diterima' in text or 'gugatan tidak diterima' in text:
        return 'ditolak'
    return 'lainnya'

df['label_putusan'] = df.apply(label_putusan, axis=1)
def kategori_hukuman_final(row):
    label = str(row['label_putusan']).lower()

    # Langsung pakai label
    if label in ['seumur_hidup', 'hukuman mati']:
        return 'berat'
    elif label in ['bebas', 'dilepaskan', 'lepas']:
        return 'bebas'

    # Berdasarkan lama hukuman
    lama = row['lama_hukuman_bulan']
    if pd.notnull(lama):
        if lama > 60:
            return 'berat'
        elif lama > 12:
            return 'sedang'
        elif lama > 0:
            return 'ringan'

    return 'lainnya'

# Tambahkan kolom kategori_hukuman ke DataFrame
df['kategori_hukuman'] = df.apply(kategori_hukuman_final, axis=1)

# Tampilkan hasil
df[['case_id', 'lama_hukuman_bulan', 'label_putusan', 'kategori_hukuman']].head(10)



,case_id,lama_hukuman_bulan,label_putusan,kategori_hukuman
0,case_001,168.0,ditahan,berat
1,case_002,120.0,ditahan,berat
2,case_003,999.0,seumur_hidup,berat
3,case_004,NaN,ditolak,lainnya
4,case_005,96.0,ditahan,berat
5,case_006,180.0,ditahan,berat
6,case_007,168.0,ditahan,berat
7,case_008,180.0,ditahan,berat
8,case_009,168.0,ditahan,berat
9,case_010,180.0,ditahan,berat


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import json

# Gunakan kolom 'text_full' dan pastikan tidak kosong
df['text_full'] = df['text_full'].fillna('')

# Split data: 80% train, 20% test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# TF-IDF vektor
stopwords_indonesia = [
    'yang', 'dan', 'di', 'ke', 'dari', 'akan', 'karena', 'bahwa', 'untuk', 'dengan',
    'pada', 'adalah', 'itu', 'ini', 'dalam', 'tidak', 'oleh', 'sebagai', 'juga', 'atau',
    'sudah', 'sangat', 'masih', 'lagi', 'lebih', 'hanya', 'maka', 'bagi', 'antara'
]

tfidf_vectorizer = TfidfVectorizer(stop_words=stopwords_indonesia, max_features=1000)
tfidf_matrix_train = tfidf_vectorizer.fit_transform(train_df['text_full'])
train_vectors = tfidf_matrix_train
train_case_ids = train_df['case_id'].tolist()


In [21]:
from typing import List

def retrieve(query: str, k: int = 5) -> List[str]:
    query_vec = tfidf_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, train_vectors).flatten()
    top_k_idx = similarities.argsort()[::-1][:k]
    top_k_case_ids = [train_case_ids[i] for i in top_k_idx]
    return top_k_case_ids


In [22]:
# Ambil 5 data dari test_df sebagai query uji
sample_queries = test_df.sample(5, random_state=42)

query_cases = []
for i, row in sample_queries.iterrows():
    q_text = row['text_full']
    q_id = row['case_id']
    top_cases = retrieve(q_text, k=5)
    query_cases.append({
        "query_id": f"Q{i+1}",
        "query_text": q_text[:250],  # ringkasan teks
        "top_5_case_ids": top_cases,
        "ground_truth": q_id
    })




In [23]:
import pandas as pd

# Ubah list of dicts menjadi DataFrame
df_queries = pd.DataFrame(query_cases)

# Tampilkan 5 hasil teratas
pd.set_option('display.max_colwidth', None)  # agar teks panjang terlihat
print("📋 Hasil Retrieval (Top-5):")
display(df_queries[['query_id', 'query_text', 'top_5_case_ids', 'ground_truth']])


📋 Hasil Retrieval (Top-5):


,query_id,query_text,top_5_case_ids,ground_truth
0,Q115,,"[case_103, case_271, case_107, case_072, case_189]",case_115
1,Q302,Direktori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia,"[case_129, case_124, case_050, case_013, case_279]",case_302
2,Q289,Direktori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia,"[case_236, case_288, case_067, case_217, case_235]",case_289
3,Q199,Direktori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia,"[case_194, case_005, case_142, case_163, case_042]",case_199
4,Q64,Direktori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia,"[case_052, case_217, case_067, case_179, case_042]",case_064


In [24]:
# 7. Simpan hasil jika perlu
output_path = '/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval/cases_labeled.csv'
df.to_csv(output_path, index=False)

In [25]:
# Simpan ke /data/eval/queries.json
import os

eval_path = '/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval'
os.makedirs(eval_path, exist_ok=True)
query_file = os.path.join(eval_path, 'queries.json')

with open(query_file, 'w', encoding='utf-8') as f:
    json.dump(query_cases, f, ensure_ascii=False, indent=2)

print(f"✅ Query uji berhasil disimpan di: {query_file}")


✅ Query uji berhasil disimpan di: /content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval/queries.json
